# M37 — Make an LLM Call Tools Reliably

**Objective:** design tool interfaces and validate reliable tool use.

M32 already handed an inference-provider contract. The useful whole here is a
**validated tool call**, not another decoder and not a multi-step agent:

`intent → model-call fixture → parse → select → validate → permission / idempotency → execute → structured result`

A parseable JSON blob is not a reliable tool call. Invalid arguments must fail
closed **before** the tool runs. This three-tool fixture is **not a production**
gateway. RAG, Qdrant, sampling labs, and LangGraph state machines stay closed.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a tool name, a `validation.ok` boolean, an
`execution_reached` flag, an `effect_count`, an `error_kind`, or a retry status.

Do not coerce wrong types. Do not treat a fluent model string as permission to
mutate state. The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M37" / "tool_runtime.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M32.inference_adaptation import InferenceConfig
from missions.M37.tool_runtime import (
    CATALOG,
    HANDOFF,
    INVALID_FIXTURES,
    MAX_ATTEMPTS,
    REGISTRY_ID,
    RUNTIME_VERSION,
    SCALE_LIMIT,
    SEED,
    SYSTEM_MAP,
    TRACE_STAGES,
    VAT_AMOUNT,
    VAT_RATE,
    VAT_TAX,
    VAT_TOTAL,
    LiveAdapterUnavailable,
    RuntimeSession,
    attach_inference_evidence,
    default_registry,
    evaluate_selection,
    execute_tool,
    handoff_contract,
    make_tool_config,
    observability_report,
    optional_live_propose,
    parse_proposal,
    pipeline_with_defect,
    propose_for_intent,
    repair_run,
    result_as_json,
    run_tool_call,
    sticky_invalid_repairer,
    validate_arguments,
    validate_proposal,
    vat_fill_repairer,
)

print("repository root:", ROOT)
print("runtime version:", RUNTIME_VERSION)
print("registry:", REGISTRY_ID)
print("seed:", SEED)
print("scale:\n", SCALE_LIMIT)


## M32 → M37 boundary: config in, tool execution out

M32 named tools as the lever for deterministic computation. It did not execute
them. This mission consumes `InferenceConfig` with frozen weights
(`training_time=False`, `weights_updated=False`) and opens the execution
boundary. Sampling stays recorded, not re-taught.


## Frozen teaching fixtures

Controls that stay fixed unless a cell names the one change:

- VAT amount `80`, rate `0.25` (tax and total are hand-computable)
- catalog `SKU-7` at price `42.0`, plus two neighbor SKUs
- side-effecting `post_ledger_entry` requiring approval and `idempotency_key`
- `max_attempts = 3`
- local model-call fixtures, not a live LLM
- sources named `anthropic-agents` and `langgraph-docs` (not imported as SDKs)


In [ ]:
registry = default_registry()
print("tools:", registry.names())
print("vat schema:")
print(json.dumps(registry.public_schemas()["compute_vat"], indent=2))
print("catalog SKU-7:", CATALOG["SKU-7"])
print("trace stages:", TRACE_STAGES)

cfg = make_tool_config()
print("InferenceConfig module:", type(cfg).__module__)
print("training_time", cfg.training_time)
evidence = attach_inference_evidence(cfg)
print("weights_updated", evidence["weights_updated"])
print("fingerprint", evidence["fingerprint"])
assert isinstance(cfg, InferenceConfig)
assert cfg.training_time is False
assert evidence["weights_updated"] is False

try:
    optional_live_propose("compute VAT", cfg)
except LiveAdapterUnavailable as exc:
    print("live adapter:", type(exc).__name__)


The attached M32 config is lineage and reproducibility evidence, not a hidden
model download. The registry is three local callables with public schemas.
A live proposer is allowed to exist and is required to fail closed here.


## Predict before running — whole tool call

Timestamp a prediction before `run-whole`.

Declared intent: compute VAT on amount 80 at rate 0.25. A local fixture, not a
live model, proposes `compute_vat`.

Predict:

- the tax and the total
- whether `execution_reached` is true
- which four trace stages appear
- whether `weights_updated` stays false


In [ ]:
print(SYSTEM_MAP)
whole = run_tool_call("vat_on_80_at_025")
print("status", whole.result.status)
print("output", whole.result.output)
print("stages", whole.stages)
print("execution_reached", whole.execution_reached)
print("weights_updated", whole.weights_updated)
print("training_time", whole.inference["training_time"])
print("handoff\n", HANDOFF)


The map ends at a structured result plus a staged `ToolTrace`. Retrieval packs,
vector indexes, and persistent agent graphs do not appear. Arithmetic that a
frozen checkpoint should not guess belongs in a tool.


## Predict before running — strict schema

Timestamp a prediction before `run-schema`.

Change only the proposed arguments. The `compute_vat` implementation stays
fixed.

Predict, for valid, missing `rate`, string `amount`, extra `currency`, rate
`1.5`, and boolean `amount`:

- which proposals are valid
- which issue kinds fire (`missing`, `wrong_type`, `extra`, `constraint`)
- whether a JSON boolean is accepted as a number


In [ ]:
spec = registry.get("compute_vat")
schema_cases = (
    ("valid", {"amount": VAT_AMOUNT, "rate": VAT_RATE}),
    ("missing_rate", {"amount": VAT_AMOUNT}),
    ("wrong_type", {"amount": "eighty", "rate": VAT_RATE}),
    ("extra_field", {"amount": VAT_AMOUNT, "rate": VAT_RATE, "currency": "USD"}),
    ("unsafe_rate", {"amount": VAT_AMOUNT, "rate": 1.5}),
    ("bool_amount", {"amount": True, "rate": VAT_RATE}),
)
schema_rows = []
for label, args in schema_cases:
    result = validate_arguments(spec, args)
    kinds = tuple(issue.kind for issue in result.issues)
    schema_rows.append((label, result.ok, kinds, result.repairable))
    print(label, "ok", result.ok, "kinds", kinds, "repairable", result.repairable)


Strict schemas reject extras, missing required keys, wrong types, out-of-range
rates, and Python `True` posing as `1`. Repairable here means missing or extra
keys, not unsafe coercion of `"eighty"` into `80`.


## Predict before running — model-call fixtures

Timestamp a prediction before `run-fixtures`.

Inject valid and invalid **model-call fixtures** (dictionaries and one
malformed JSON string). No live model.

Predict:

- the status of a valid VAT fixture
- whether malformed JSON is a parse/schema failure or a tool failure
- whether an unknown tool name reaches a handler


In [ ]:
fixture_traces = {}
for name in ("missing_rate", "wrong_type", "extra_field", "malformed_json", "unknown_tool"):
    fixture_traces[name] = run_tool_call(INVALID_FIXTURES[name])
    t = fixture_traces[name]
    print(name, t.result.status, t.error_kind, "reached", t.execution_reached)
valid_fixture = run_tool_call(propose_for_intent("vat_on_80_at_025"))
print("valid", valid_fixture.result.status, valid_fixture.result.output)
print("unknown issues", tuple(issue.kind for issue in fixture_traces["unknown_tool"].validation.issues))


Fixtures stand in for a model. The runtime must treat them as untrusted input.
Parse failures and unknown tool names stop in validation, not in a handler.


## Predict before running — validate before execution

Timestamp a prediction before `run-validation`.

One named change: feed invalid proposals through `run_tool_call` on a shared
`RuntimeSession`, then one valid VAT call on the same session.

Predict:

- `session.execution_count` after the invalid batch
- whether any invalid trace has `execution_reached`
- `session.execution_count` after the valid call


In [ ]:
session = RuntimeSession()
invalid_batch = []
for name in ("missing_rate", "wrong_type", "extra_field", "unsafe_rate"):
    invalid_batch.append(run_tool_call(INVALID_FIXTURES[name], session=session))
print("executions after invalids", session.execution_count)
print("invalid reached", [t.execution_reached for t in invalid_batch])
valid_after = run_tool_call("vat_on_80_at_025", session=session)
print("executions after valid", session.execution_count)
print("valid reached", valid_after.execution_reached)


The execution counter is the discriminator. If an invalid proposal can increment
it, the trust boundary is already gone — fluent JSON does not matter.


## Predict before running — structured results

Timestamp a prediction before `run-results`.

Same tools. Compare a successful VAT result with a missing-field error.

Predict:

- whether both serialize as JSON objects
- which `error_kind` the missing-field path carries
- whether the success payload includes `tax` and `total`


In [ ]:
success = run_tool_call("vat_on_80_at_025")
schema_err = run_tool_call(INVALID_FIXTURES["missing_rate"])
print("success json", result_as_json(success.result))
print("schema json", result_as_json(schema_err.result))
print("kinds", success.result.error_kind, schema_err.result.error_kind)
print("error types", success.result.error_type, schema_err.result.error_type)


Structured outputs are not a pretty printer. Status, error kind, and payload
must be distinct so M38 can branch on them without parsing prose.


## Predict before running — tool selection

Timestamp a prediction before `run-selection`.

Three intents, schemas held fixed:

1. VAT on 80 at 0.25
2. catalog price of SKU-7
3. write a haiku about autumn rain

Predict the selected tool for each, including the no-tool case, and whether
the haiku reaches execution.


In [ ]:
selection = evaluate_selection()
print(selection)
haiku = run_tool_call("write_a_haiku")
sku = run_tool_call("price_of_sku_7")
print("haiku", haiku.result.status, haiku.execution_reached, haiku.selected_tool)
print("sku", sku.result.output, sku.selected_tool)


Selection is non-trivial only when at least two tools exist and refusing to
call a tool is a legal outcome. A haiku is not VAT and not a catalog lookup.


## Predict before running — approval and idempotency

Timestamp a prediction before `run-idempotency`.

Same ledger, same key `vat-80-0.25`. First call unapproved, then approved,
then a replay with the same key.

Predict:

- `effect_count` after the unapproved call
- `effect_count` after the first approved post
- `effect_count` after the replay, and whether `replayed` is true


In [ ]:
ledger_session = RuntimeSession()
denied = run_tool_call("post_vat_to_ledger", session=ledger_session, approved=False)
print("denied", denied.result.status, "effect", ledger_session.ledger.effect_count, "reached", denied.execution_reached)
posted = run_tool_call("post_vat_to_ledger", session=ledger_session, approved=True)
print("posted", posted.replayed, posted.result.output, "effect", ledger_session.ledger.effect_count)
replayed = run_tool_call("post_vat_to_ledger", session=ledger_session, approved=True)
print("replayed", replayed.replayed, "effect", ledger_session.ledger.effect_count, "executions", ledger_session.execution_count)


Approval is a gate, not a log line. Idempotency is a store consulted **after**
validation and **inside** execution. Replay must not append a second row.


## Predict before running — bounded repair retry

Timestamp a prediction before `run-retry`.

Call fixture fixed: missing `rate`. One path uses a sticky repairer that
returns the same invalid proposal. The other fills `rate` from a known table
without coercing types.

Predict:

- status, attempt count, and `execution_reached` for the sticky path
- whether the fill path succeeds, and on which attempt


In [ ]:
sticky = run_tool_call(
    INVALID_FIXTURES["missing_rate"],
    max_attempts=MAX_ATTEMPTS,
    repairer=sticky_invalid_repairer,
)
print(
    "sticky",
    sticky.result.status,
    "attempts",
    len(sticky.attempts),
    "remaining",
    sticky.retry_budget_remaining,
    "reached",
    sticky.execution_reached,
)
filled = run_tool_call(
    INVALID_FIXTURES["missing_rate"],
    max_attempts=MAX_ATTEMPTS,
    repairer=vat_fill_repairer,
)
print("filled", filled.result.status, "attempts", len(filled.attempts), filled.result.output)


Retries are a bound, not a hope. A repairer that cannot change the proposal
must stop at `retry_exhausted` without ever calling the tool. Filling a missing
field is allowed; turning `"eighty"` into `80` is not.


## Predict before running — schema failure vs tool failure

Timestamp a prediction before `run-tool-error`.

Arguments for `lookup_catalog_price` with `SKU-ZZ` match the SKU pattern but
the catalog has no such item. Compare that with a VAT call missing `rate`.

Predict:

- `error_kind` and `execution_reached` for the missing-rate call
- `error_kind`, `error_type`, `validation.ok`, and `execution_reached` for `SKU-ZZ`


In [ ]:
schema_miss = run_tool_call(INVALID_FIXTURES["missing_rate"])
tool_miss = run_tool_call(INVALID_FIXTURES["unknown_sku"])
print(
    "schema",
    schema_miss.result.error_kind,
    schema_miss.result.error_type,
    "reached",
    schema_miss.execution_reached,
    "validation",
    schema_miss.validation.ok,
)
print(
    "tool",
    tool_miss.result.error_kind,
    tool_miss.result.error_type,
    "reached",
    tool_miss.execution_reached,
    "validation",
    tool_miss.validation.ok,
    tool_miss.result.message,
)


If validation passed and the handler failed, that is a tool error. If
validation failed, the handler must not have run. Mixing the two labels makes
M38 retry the wrong layer.


In [ ]:
labels = ["valid VAT", "missing rate", "wrong type", "unknown SKU"]
traces_for_plot = [
    run_tool_call("vat_on_80_at_025"),
    run_tool_call(INVALID_FIXTURES["missing_rate"]),
    run_tool_call(INVALID_FIXTURES["wrong_type"]),
    run_tool_call(INVALID_FIXTURES["unknown_sku"]),
]
reached = []
for trace in traces_for_plot:
    if trace.execution_reached:
        reached.append(3)
    elif trace.selected_tool is not None:
        reached.append(2)
    else:
        reached.append(1)
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2a9d8f", "#e76f51", "#e76f51", "#e9c46a"]
ax.bar(np.arange(len(labels)), reached, color=colors)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["selected", "validated", "executed"])
ax.set_ylabel("farthest gate reached")
ax.set_xlabel("model-call fixture")
ax.set_title("Which proposals reached execute_tool?")
ax.set_ylim(0, 3.5)
fig.tight_layout()
plt.show()
print("plot asks: which proposals reached execute_tool? unknown SKU should; missing rate should not")
print("reached ranks", list(zip(labels, reached)))


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `run_tool_call`, `validate_arguments`, `execute_tool`, and `repair_run`
in `missions/M37/tool_runtime.py`. Dump `inspect.getsource` of the wrapper
and probe live objects (error types, `effect_count`, retry budget).

Predict:

- whether `run_tool_call` calls `validate_arguments` before `execute_tool`
- what `execute_tool` does with an approved replay of the same key
- what `repair_run` reuses from a broken object
- the live `error_type` of a wrong-type VAT fixture and the `effect_count`
  after an approved ledger replay


In [ ]:
wrapper_src = inspect.getsource(run_tool_call)
validate_src = inspect.getsource(validate_arguments)
execute_src = inspect.getsource(execute_tool)
repair_src = inspect.getsource(repair_run)
print(wrapper_src)
print("--- validate_arguments (first lines) ---")
for line in validate_src.splitlines()[:15]:
    print(line)
print("--- execute_tool ---")
print(execute_src)
print("--- repair_run ---")
print(repair_src)

invalid_type = run_tool_call(INVALID_FIXTURES["wrong_type"])
print("wrong_type error_type", invalid_type.result.error_type)
print("wrong_type execution_reached", invalid_type.execution_reached)
print("wrong_type issue kinds", tuple(issue.kind for issue in invalid_type.validation.issues))

probe_session = RuntimeSession()
first_post = run_tool_call("post_vat_to_ledger", session=probe_session, approved=True)
second_post = run_tool_call("post_vat_to_ledger", session=probe_session, approved=True)
print("effect_count", probe_session.ledger.effect_count)
print("second_post replayed", second_post.replayed)
print("sticky retry_budget_remaining", sticky.retry_budget_remaining)
print("unknown sku error_type", tool_miss.result.error_type)
print("unknown sku execution_reached", tool_miss.execution_reached)
report = observability_report(whole)
print("handoff", report["handoff"])
print("trace fields", report["trace_fields"])
print("weights_updated on live trace", whole.weights_updated)


## Predict before running — Controlled failure: malformed side effect

Timestamp a prediction before `run-failure`.

A side-effecting ledger post is proposed with a non-numeric amount. The
teaching registry and VAT schema stay fixed. One named defect removes a gate.

Predict:

- whether `effect_count` increments
- whether a healthy validator would accept the same arguments
- whether you should reach for a bigger model first


In [ ]:
broken_malformed = pipeline_with_defect(defect="malformed_reaches_side_effect")
print("defect", broken_malformed.defect, "claim", broken_malformed.claim)
print("effect_count", broken_malformed.effect_count)
print("execution_reached", broken_malformed.execution_reached)
print("validation_bypassed", broken_malformed.validation_bypassed)
print("amount_type", broken_malformed.audit["amount_type"])
print("healthy_validation_ok", broken_malformed.audit["healthy_validation_ok"])
print("session executions", broken_malformed.session_execution_count)


## Predict before running — Controlled failure: timeout retry

Timestamp a prediction before `run-failure-duplicate`.

A valid, approved ledger post is retried after a simulated timeout. The
proposal itself does not change. One named defect removes a different gate.

Predict:

- `effect_count` after the retry
- whether the two entry ids match
- which gate would have made the second call a replay instead of a second post


In [ ]:
broken_dup = pipeline_with_defect(defect="duplicate_side_effect")
print("defect", broken_dup.defect, "claim", broken_dup.claim)
print("effect_count", broken_dup.effect_count)
print("idempotency_consulted", broken_dup.idempotency_consulted)
print("entry ids", broken_dup.audit["first_entry_id"], broken_dup.audit["second_entry_id"])
print("simulated", broken_dup.audit["simulated"])


## Diagnosis record (your log, not this repo)

Record symptom, hypotheses (missing validation vs missing idempotency vs
tool bug), a discriminating measurement (`healthy_validation_ok`,
`effect_count`, entry ids), and only then the root cause. Do not paste the
filled diagnosis into this notebook.


## Predict before running — repair from the broken traces

Timestamp a prediction before `run-failure-repair`.

`repair_run` is called on the broken objects already in memory. It must reuse
those proposals and the initial ledger snapshots.

Predict:

- repaired malformed `execution_reached` and `effect_count`
- repaired duplicate `effect_count` and whether the second call is a replay
- whether the original broken traces still show the defective counts


In [ ]:
repaired_malformed = repair_run(broken_malformed)
repaired_dup = repair_run(broken_dup)
print(
    "malformed repaired",
    repaired_malformed.result_trace.result.status,
    "effect",
    repaired_malformed.effect_count,
    "reached",
    repaired_malformed.execution_reached,
)
print(
    "dup repaired effect",
    repaired_dup.effect_count,
    "second_replayed",
    repaired_dup.audit["second_replayed"],
)
print("broken still", broken_malformed.effect_count, broken_dup.effect_count)
print("malformed still bypassed", broken_malformed.validation_bypassed)
print("dup still consulted", broken_dup.idempotency_consulted)


Repair did not invent a second unrelated happy-path run from scratch and did
not open a state machine. It reused the broken object's proposal and initial
ledger snapshot. The defective traces still post: that is the regression.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- VAT tax/total on the teaching fixture
- schema table (valid / missing / wrong-type / extra / constraint / bool)
- execution counter after invalid fixtures
- selection table for two tools plus no-tool
- structured success vs schema vs tool JSON
- unapproved post, approved post, same-key replay
- sticky retry exhaustion and a successful fill repair
- malformed and duplicate diagnosis plus `repair_run`

See `missions/M37/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M37/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Compute duty on a fresh `(value, rate)` pair, classify four proposed calls,
state the idempotency count for a refund retry, name an error locus, and
explain why parseable JSON is not sufficient reliability.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M37/adr_prompt.md` to choose a V10 tool-execution trust
boundary (schema strictness, permissions, idempotency, retry limits, trace
fields). Do not claim a production gateway and do not implement a
state machine, memory router, or eval harness here.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M37 for the package, not as a substitute for the learner ADR.


## M32 → M37 → M38 handoff

M32 supplied `InferenceConfig` and named tools as a lever. M37 executed that
lever behind a schema, an approval gate, and an idempotency key.

M38 may wrap this runtime in an explicit state machine. It must consume the
registry and `ToolTrace`. It must not relabel a schema failure as a tool
failure, and it must not replay a side effect because chat history "looked"
complete.

Reusable artifacts: default registry, `run_tool_call`, staged traces,
typed errors, and `handoff_contract()`.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why is parseable JSON not sufficient reliability for a tool call?
2. What identity tells you a schema failure never reached the tool?
3. Why can a valid SKU still be a tool error?
4. When is a timeout-retry allowed to post a second ledger row?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert whole.result.output["tax"] == VAT_TAX and whole.result.output["total"] == VAT_TOTAL
assert cfg.training_time is False
assert isinstance(cfg, InferenceConfig)
assert type(cfg).__module__ == "missions.M32.inference_adaptation"
assert evidence["weights_updated"] is False
assert session.execution_count == 1
assert all(not trace.execution_reached for trace in invalid_batch)
assert selection["n_correct"] == 3
assert haiku.result.status == "no_tool" and haiku.execution_reached is False
assert ledger_session.ledger.effect_count == 1 and replayed.replayed
assert denied.result.status == "permission_denied"
assert sticky.result.status == "retry_exhausted" and sticky.execution_reached is False
assert filled.result.status == "success" and len(filled.attempts) == 2
assert schema_miss.result.error_kind == "schema" and tool_miss.result.error_kind == "tool"
assert tool_miss.execution_reached and not schema_miss.execution_reached
assert broken_malformed.effect_count == 1 and repaired_malformed.effect_count == 0
assert broken_dup.effect_count == 2 and repaired_dup.effect_count == 1
assert broken_malformed.validation_bypassed and not repaired_malformed.validation_bypassed
assert probe_session.ledger.effect_count == 1
print("M37 integrity checks passed")
print(handoff_contract()["handoff"])
